# **Лабораторна робота №2**
## з дисципліни **"Обробка надвеликих масивів даних"**

##### *За темою:* **"Робота з графовими структурами"**

<div style="display: flex;">

<div style="flex: 1;">
    <i>Виконав:</i>
    <br><br>
    <i><b>Гавлицький Іван КІ-61мп</b></i>
</div>

<div style="flex: 1;">
    <i>Прийняв:</i>
    <br><br>
    <i><b>Пишнограєв<br>Іван Олександрович</b></i>
</div>

</div>

Варіант 3: Німеччина – Велика Британія.

## Підготовка

In [1]:
import sys
from pathlib import Path

import pandas as pd
from graphframes import GraphFrame
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, StringType, StructField, StructType
from pyspark.sql.window import Window

from lab002.cluster import compose, compose_up, run
from lab002.paths import simple_paths
from lab002.spark import spark_session

COMPOSE_FILE = Path("infra/docker-compose.yml")
assert COMPOSE_FILE.exists(), "run the notebook from labs/002-spark-graphframes"
DATA_DIR = Path("data").resolve()
# Bind-mounted into the Spark containers; must exist before `compose up`, or Docker creates it owned by root.
DATA_DIR.mkdir(exist_ok=True)
OPENFLIGHTS = DATA_DIR / "openflights"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)


def show(df, n=20):
    return df.limit(n).toPandas()

## Запуск кластера

Spark 4.0.4 у Docker: master і два workers (`infra/`). Драйвер — цей ноутбук.

In [2]:
compose_up(COMPOSE_FILE, DATA_DIR)
compose(COMPOSE_FILE, DATA_DIR, "ps", "--format", "table {{.Name}}\t{{.Status}}");

$ docker compose --progress quiet -f infra/docker-compose.yml up -d --build --wait --wait-timeout 300
$ docker compose --progress quiet -f infra/docker-compose.yml ps --format 'table {{.Name}}	{{.Status}}'
NAME             STATUS
spark-master     Up 10 seconds (healthy)
spark-worker-1   Up 7 seconds (healthy)
spark-worker-2   Up 6 seconds (healthy)


In [3]:
spark = spark_session(DATA_DIR)
spark.sparkContext.master, spark.sparkContext.defaultParallelism

:: loading settings :: url = jar:file:/home/nuinashco/Projects/iasa-bigdata/.venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/nuinashco/.ivy2.5.2/cache
The jars for the packages stored in: /home/nuinashco/.ivy2.5.2/jars
io.graphframes#graphframes-spark4_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5f9aa23c-36ea-4ca8-af2e-b338c4fe33b8;1.0
	confs: [default]


	found io.graphframes#graphframes-spark4_2.13;0.12.3 in central
	found io.graphframes#graphframes-graphx-spark4_2.13;0.12.3 in central
	found org.apache.datasketches#datasketches-java;6.2.0 in central
	found org.apache.datasketches#datasketches-memory;3.0.2 in central
:: resolution report :: resolve 222ms :: artifacts dl 8ms
	:: modules in use:
	io.graphframes#graphframes-graphx-spark4_2.13;0.12.3 from central in [default]
	io.graphframes#graphframes-spark4_2.13;0.12.3 from central in [default]
	org.apache.datasketches#datasketches-java;6.2.0 from central in [default]
	org.apache.datasketches#datasketches-memory;3.0.2 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   4   |   0   |   0   |   0   ||   4   |  

26/10/08 00:24:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


('spark://127.0.0.1:7077', 8)

## Завдання 1. Підготовка даних та розрахунок дистанцій

### Завантаження даних

Набори даних [OpenFlights](https://openflights.org/data.php): аеропорти, авіакомпанії та маршрути.

In [4]:
run([sys.executable, "scripts/download_data.py"]);

$ /home/nuinashco/Projects/iasa-bigdata/.venv/bin/python scripts/download_data.py
[skip] airports-extended.dat already present
[skip] airlines.dat already present
[skip] routes.dat already present


### Читання та очищення

Файли без заголовків, тому схеми задано явно; відсутні значення в OpenFlights позначено `\N`.

In [5]:
S, I, D = StringType(), IntegerType(), DoubleType()


def schema(*fields):
    return StructType([StructField(name, dtype) for name, dtype in fields])


AIRPORTS = schema(
    ("id", I), ("name", S), ("city", S), ("country", S), ("iata", S), ("icao", S), ("latitude", D),
    ("longitude", D), ("altitude", I), ("timezone", D), ("dst", S), ("tz", S), ("type", S), ("source", S),
)
AIRLINES = schema(
    ("id", I), ("name", S), ("alias", S), ("iata", S), ("icao", S), ("callsign", S), ("country", S), ("active", S),
)
ROUTES = schema(
    ("airline", S), ("airline_id", I), ("src", S), ("src_id", I), ("dst", S), ("dst_id", I), ("codeshare", S),
    ("stops", I), ("equipment", S),
)


def read(name, file_schema):
    # Absolute path: executors read the file themselves, and data/ is mounted at this path on every worker.
    return spark.read.csv(str(OPENFLIGHTS / name), schema=file_schema, nullValue="\\N")


airports = read("airports-extended.dat", AIRPORTS)
airlines = read("airlines.dat", AIRLINES)
routes = read("routes.dat", ROUTES)
for name, df in [("airports", airports), ("airlines", airlines), ("routes", routes)]:
    df.createOrReplaceTempView(name)

pd.DataFrame({"rows": [airports.count(), airlines.count(), routes.count()]}, index=["airports", "airlines", "routes"])

,rows
airports,12668
airlines,6162
routes,67663


In [6]:
show(routes, 5)

,airline,airline_id,src,src_id,dst,dst_id,codeshare,stops,equipment
0,2B,410,AER,2965,KZN,2990,None,0,CR2
1,2B,410,ASF,2966,KZN,2990,None,0,CR2
2,2B,410,ASF,2966,MRV,2962,None,0,CR2
3,2B,410,CEK,2968,KZN,2990,None,0,CR2
4,2B,410,CEK,2968,OVB,4078,None,0,CR2


Якість маршрутів: без ідентифікатора аеропорту чи авіакомпанії, codeshare-рейси та петлі:

In [7]:
spark.sql(r"""
SELECT
    COUNT(*) AS routes,
    COUNT_IF(src_id IS NULL OR dst_id IS NULL) AS without_airport_id,
    COUNT_IF(airline_id IS NULL) AS without_airline_id,
    COUNT_IF(codeshare = 'Y') AS codeshare,
    COUNT_IF(src_id = dst_id) AS same_airport

FROM
    routes
""").toPandas()

,routes,without_airport_id,without_airline_id,codeshare,same_airport
0,67663,423,479,14597,1


### Відстані (формула гаверсинуса)

Join маршрутів з аеропортами вильоту та прильоту додає координати; маршрути без ідентифікатора аеропорту та петлі відкидаються. Відстань по великому колу:

$d = 2R \arcsin\sqrt{\sin^2\frac{\varphi_2 - \varphi_1}{2} + \cos\varphi_1 \cos\varphi_2 \sin^2\frac{\lambda_2 - \lambda_1}{2}}$, $R = 6371$ км.

In [8]:
spark.sql(r"""
CREATE OR REPLACE TEMP VIEW flights_all AS
SELECT
    r.airline,
    r.airline_id,
    r.codeshare,
    r.src_id,
    r.dst_id,
    s.iata AS src_iata,
    d.iata AS dst_iata,
    s.country AS src_country,
    d.country AS dst_country,
    2 * 6371 * ASIN(SQRT(
        POWER(SIN(RADIANS(d.latitude - s.latitude) / 2), 2)
        + COS(RADIANS(s.latitude)) * COS(RADIANS(d.latitude))
        * POWER(SIN(RADIANS(d.longitude - s.longitude) / 2), 2)
    )) AS distance_km

FROM
    routes AS r

INNER JOIN
    airports AS s
    ON r.src_id = s.id

INNER JOIN
    airports AS d
    ON r.dst_id = d.id

WHERE 1=1
    AND r.src_id != r.dst_id
""")

spark.sql(r"""
SELECT
    airline,
    src_id,
    src_iata,
    dst_id,
    dst_iata,
    ROUND(distance_km) AS distance_km

FROM
    flights_all

ORDER BY distance_km DESC

LIMIT 6
""").toPandas()

,airline,src_id,src_iata,dst_id,dst_iata,distance_km
0,P0,907,LUN,5613,NaN,16082.0
1,P0,5613,NaN,907,LUN,16082.0
2,P0,910,NLA,5613,NaN,15937.0
3,P0,5613,NaN,910,NLA,15937.0
4,AA,3361,SYD,3670,DFW,13808.0
5,QF,3361,SYD,3670,DFW,13808.0


Найдовші «рейси» (понад 15 900 км, довше за будь-який реальний безпосадковий рейс) ведуть до аеропорту 5613. Його запис пошкоджений: місто й країна — Солвезі, Замбія, а назва, код ICAO та координати належать аеродрому в Каліфорнії. Маршрути через цей аеропорт виключено; наступні за довжиною маршрути (SYD → DFW, 13 808 км) реальні.

In [9]:
BROKEN_AIRPORTS = (5613,)

show(airports.filter(F.col("id").isin(*BROKEN_AIRPORTS)).select("id", "name", "city", "country", "icao", "latitude", "longitude", "tz"))

,id,name,city,country,icao,latitude,longitude,tz
0,5613,Los Alamitos Army Air Field,Solwesi,Zambia,KSLI,33.790001,-118.052002,America/Los_Angeles


In [10]:
broken_ids = ", ".join(map(str, BROKEN_AIRPORTS))
spark.sql(rf"""
CREATE OR REPLACE TEMP VIEW flights AS
SELECT
    *

FROM
    flights_all

WHERE 1=1
    AND src_id NOT IN ({broken_ids})
    AND dst_id NOT IN ({broken_ids})
""")

spark.sql(r"""
SELECT
    COUNT(*) AS flights,
    ROUND(MIN(distance_km)) AS min_km,
    ROUND(AVG(distance_km)) AS avg_km,
    ROUND(MAX(distance_km)) AS max_km

FROM
    flights
""").toPandas()

,flights,min_km,avg_km,max_km
0,67235,1.0,1845.0,13808.0


### Сумарна довжина рейсів авіакомпаній

Враховано лише маршрути, які авіакомпанія виконує сама: codeshare-маршрут (`codeshare = 'Y'`) — це рейс іншого перевізника, що продається під її кодом. Кожен маршрут рахується один раз, незалежно від частоти рейсів.

In [11]:
AIRLINE_TOTALS = r"""
SELECT
    a.name AS airline,
    a.country,
    COUNT(*) AS routes,
    ROUND(SUM(f.distance_km)) AS total_km

FROM
    flights AS f

INNER JOIN
    airlines AS a
    ON f.airline_id = a.id

WHERE 1=1
    AND f.codeshare IS NULL

GROUP BY a.id, a.name, a.country

ORDER BY total_km {order}

LIMIT 10
"""

spark.sql(AIRLINE_TOTALS.format(order="DESC")).toPandas()

,airline,country,routes,total_km
0,Ryanair,Ireland,2484,3700425.0
1,US Airways,United States,1446,3477460.0
2,American Airlines,United States,1265,3342406.0
3,United Airlines,United States,905,3046248.0
4,Delta Air Lines,United States,1146,3013769.0
5,China Southern Airlines,China,1238,1733470.0
6,easyJet,United Kingdom,1130,1582234.0
7,Southwest Airlines,United States,1098,1542333.0
8,Air China,China,786,1521810.0
9,Lufthansa,Germany,507,1513728.0


In [12]:
spark.sql(AIRLINE_TOTALS.format(order="ASC")).toPandas()

,airline,country,routes,total_km
0,TransHolding System,Brazil,2,36.0
1,SOCHI AIR CHATER,Russia,2,38.0
2,Air Seychelles,Seychelles,2,87.0
3,LACSA,Costa Rica,2,111.0
4,Reem Air,Kyrgyzstan,2,142.0
5,Sibaviatrans,Russia,2,170.0
6,International AirLink,Jamaica,2,179.0
7,Papillon Grand Canyon Helicopters,United States,2,186.0
8,Helijet,Canada,2,198.0
9,40-Mile Air,United States,2,237.0


## Завдання 2. Пошук маршрутів (Motifs / BFS)

### Граф авіасполучень

Вершини — аеропорти, що мають хоча б один маршрут; ребра — напрямки «аеропорт → аеропорт». Маршрути різних авіакомпаній між тією ж парою аеропортів об'єднано в одне ребро (з кількістю авіакомпаній), інакше кожен шлях повторювався б для кожної комбінації перевізників.

In [13]:
edges = spark.sql(r"""
SELECT
    src_id AS src,
    dst_id AS dst,
    FIRST(distance_km) AS distance_km,
    COUNT(DISTINCT airline_id) AS airlines

FROM
    flights

GROUP BY src_id, dst_id
""").cache()
edges.createOrReplaceTempView("edges")

vertices = spark.sql(r"""
SELECT
    a.id,
    COALESCE(a.iata, a.icao, CAST(a.id AS STRING)) AS code,
    a.name,
    a.city,
    a.country,
    a.latitude,
    a.longitude

FROM
    airports AS a

WHERE 1=1
    AND a.id IN (SELECT src FROM edges UNION SELECT dst FROM edges)
""").cache()
vertices.createOrReplaceTempView("vertices")

g = GraphFrame(vertices, edges)
pd.DataFrame({"count": [g.vertices.count(), g.edges.count()]}, index=["vertices", "edges"])

,count
vertices,3329
edges,37269


### Пошук мотивів

Усі маршрути з аеропортів Німеччини до аеропортів Великої Британії з 1, 2 і 3 сегментами (до 2 пересадок). Мотив описує ланцюжок ребер, `simple_paths` додає умови на кінцеві вершини та забороняє повторне відвідування аеропорту.

In [14]:
FROM_DE = "{v}.country = 'Germany'"
TO_UK = "{v}.country = 'United Kingdom'"

de_uk = {segments: simple_paths(g, segments, FROM_DE, TO_UK).cache() for segments in (1, 2, 3)}
pd.DataFrame({"routes": {segments: df.count() for segments, df in de_uk.items()}}).rename_axis("segments")

,routes
segments,
1,73
2,9178
3,506498


In [15]:
show(de_uk[1].orderBy("distance_km"), 10)

,segments,route,distance_km
0,1,NRN → STN,407.805873
1,1,DUS → STN,455.938911
2,1,DUS → LCY,466.072330
3,1,DUS → LGW,484.609763
4,1,CGN → STN,492.464215
5,1,DUS → LHR,501.891043
6,1,DTM → STN,509.805233
7,1,CGN → LGW,513.811456
8,1,CGN → LHR,534.220806
9,1,HHN → STN,537.439425


In [16]:
show(de_uk[2].orderBy("distance_km"), 10)

,segments,route,distance_km
0,2,DUS → AMS → NWI,417.511556
1,2,CGN → AMS → NWI,468.809446
2,2,DUS → AMS → SEN,468.980381
3,2,DUS → AMS → STN,491.224808
4,2,DUS → AMS → LCY,513.410833
5,2,CGN → AMS → SEN,520.278271
6,2,BRE → AMS → NWI,522.498312
7,2,DUS → AMS → LTN,532.194424
8,2,CGN → AMS → STN,542.522698
9,2,DUS → AMS → LGW,543.045250


In [17]:
show(de_uk[3].orderBy("distance_km"), 10)

,segments,route,distance_km
0,3,DUS → AMS → NWI → HUY,565.374510
1,3,FRA → DUS → AMS → NWI,606.379648
2,3,CGN → AMS → NWI → HUY,616.672401
3,3,SCN → LUX → AMS → NWI,634.016665
4,3,FMO → BRE → AMS → NWI,648.368344
5,3,FRA → DUS → AMS → SEN,657.848473
6,3,SCN → LUX → CDG → LGW,660.652984
7,3,DUS → AMS → BRU → LGW,664.044294
8,3,DUS → AMS → NWI → MAN,667.129396
9,3,BRE → AMS → NWI → HUY,670.361266


Найчастіші аеропорти пересадки на маршрутах з однією пересадкою:

In [18]:
show(
    de_uk[2]
    .groupBy(F.split("route", " → ")[1].alias("hub"))
    .count()
    .orderBy(F.desc("count")),
    10,
)

,hub,count
0,PMI,648
1,AGP,391
2,TFS,340
3,AMS,288
4,FAO,286
5,ALC,260
6,DUB,242
7,IBZ,240
8,ACE,240
9,LPA,238


### Пошук у ширину (BFS)

BFS повертає лише **найкоротші** за кількістю сегментів шляхи: щойно знайдено цільову вершину на певній глибині, пошук зупиняється. Оскільки між Німеччиною та Великою Британією є прямі рейси, BFS знаходить тільки їх.

In [19]:
bfs = g.bfs(fromExpr="country = 'Germany'", toExpr="country = 'United Kingdom'", maxPathLength=3)
bfs.count(), bfs.columns

(73, ['from', 'e0', 'to'])

Для аеропорту без прямих рейсів до Великої Британії BFS знаходить маршрути з однією пересадкою:

In [20]:
no_direct = spark.sql(r"""
SELECT
    v.id,
    v.code,
    v.name,
    COUNT(*) AS routes

FROM
    vertices AS v

INNER JOIN
    edges AS e
    ON e.src = v.id

WHERE 1=1
    AND v.country = 'Germany'
    AND v.id NOT IN (
        SELECT e2.src
        FROM edges AS e2
        INNER JOIN vertices AS v2
            ON e2.dst = v2.id
        WHERE v2.country = 'United Kingdom'
    )

GROUP BY v.id, v.code, v.name

ORDER BY routes DESC

LIMIT 1
""").first()

bfs_one = g.bfs(fromExpr=f"id = {no_direct.id}", toExpr="country = 'United Kingdom'", maxPathLength=3)
show(
    bfs_one.select(
        F.concat_ws(" → ", "from.code", "v1.code", "to.code").alias("route"),
        (F.col("e0.distance_km") + F.col("e1.distance_km")).alias("distance_km"),
    ).orderBy("distance_km"),
    10,
)

,route,distance_km
0,FDH → CGN → STN,887.028479
1,FDH → FRA → LCY,891.905035
2,FDH → DUS → STN,904.395832
3,FDH → CGN → LGW,908.375720
4,FDH → DUS → LCY,914.529252
5,FDH → FRA → LHR,926.272522
6,FDH → CGN → LHR,928.785070
7,FDH → DUS → LGW,933.066684
8,FDH → DUS → LHR,950.347964
9,FDH → CGN → BHX,1032.267664


In [21]:
pd.DataFrame(
    {
        "BFS": [bfs.count(), bfs_one.count()],
        "motifs (1-3 segments)": [
            sum(df.count() for df in de_uk.values()),
            sum(simple_paths(g, k, f"{{v}}.id = {no_direct.id}", TO_UK).count() for k in (1, 2, 3)),
        ],
    },
    index=["Germany → United Kingdom", f"{no_direct.code} → United Kingdom"],
)

,BFS,motifs (1-3 segments)
Germany → United Kingdom,73,515749
FDH → United Kingdom,90,6028


## Завдання 3. Найкоротший та найдовший маршрути

Між Берлін-Тегель (TXL) і Единбургом (EDI), до 3 пересадок (до 4 сегментів), довжина маршруту — сума відстаней сегментів у км.

In [22]:
SRC, DST = "TXL", "EDI"

paths = None
for segments in (1, 2, 3, 4):
    found = simple_paths(g, segments, f"{{v}}.code = '{SRC}'", f"{{v}}.code = '{DST}'")
    paths = found if paths is None else paths.unionByName(found)
paths = paths.cache()

show(paths.groupBy("segments").count().orderBy("segments"))

,segments,count
0,2,40
1,3,2745
2,4,180069


In [23]:
show(paths.orderBy("distance_km"), 5)

,segments,route,distance_km
0,2,TXL → AMS → EDI,1244.550407
1,3,TXL → AMS → NWI → EDI,1290.150128
2,3,TXL → DUS → AMS → EDI,1313.451568
3,2,TXL → CPH → EDI,1343.178733
4,2,TXL → CGN → EDI,1358.983140


In [24]:
show(paths.orderBy(F.desc("distance_km")), 5)

,segments,route,distance_km
0,4,TXL → ORD → HKG → EWR → EDI,37808.463810
1,4,TXL → JFK → HKG → EWR → EDI,37542.824146
2,4,TXL → ORD → DEL → EWR → EDI,36111.027015
3,4,TXL → ORD → PVG → EWR → EDI,35519.979974
4,4,TXL → JFK → PVG → EWR → EDI,35346.186868


## Завдання 4. Визначення впливовості (PageRank)

`resetProbability = 0.15` відповідає коефіцієнту демпфування 0,85.

In [25]:
ranks = g.pageRank(resetProbability=0.15, maxIter=20).vertices
degrees = g.inDegrees.join(g.outDegrees, "id")

show(
    ranks.join(degrees, "id")
    .select("code", "name", "city", "country", F.round("pagerank", 2).alias("pagerank"), "inDegree", "outDegree")
    .orderBy(F.desc("pagerank")),
    10,
)

,code,name,city,country,pagerank,inDegree,outDegree
0,ATL,Hartsfield Jackson Atlanta International Airport,Atlanta,United States,15.95,216,217
1,ORD,Chicago O'Hare International Airport,Chicago,United States,14.63,203,206
2,ISL,Atatürk International Airport,Istanbul,Turkey,14.62,228,225
3,DEN,Denver International Airport,Denver,United States,14.54,168,169
4,DFW,Dallas Fort Worth International Airport,Dallas-Fort Worth,United States,14.29,185,187
5,DME,Domodedovo International Airport,Moscow,Russia,13.72,188,187
6,CDG,Charles de Gaulle International Airport,Paris,France,13.44,233,237
7,FRA,Frankfurt am Main Airport,Frankfurt,Germany,13.08,238,239
8,PEK,Beijing Capital International Airport,Beijing,China,12.97,206,206
9,AMS,Amsterdam Airport Schiphol,Amsterdam,Netherlands,12.40,231,232


`ISL` — стамбульський аеропорт Ататюрк: у 2019 році код `IST` перейшов до нового аеропорту, і довідник аеропортів OpenFlights це відображає, тоді як маршрути зібрано раніше (≈ 2014).

Порівняння з центральністю за ступенем (кількість напрямків вильоту):

In [26]:
show(
    vertices.join(degrees, "id")
    .select("code", "name", "country", "inDegree", "outDegree")
    .orderBy(F.desc("outDegree")),
    10,
)

,code,name,country,inDegree,outDegree
0,FRA,Frankfurt am Main Airport,Germany,238,239
1,CDG,Charles de Gaulle International Airport,France,233,237
2,AMS,Amsterdam Airport Schiphol,Netherlands,231,232
3,ISL,Atatürk International Airport,Turkey,228,225
4,ATL,Hartsfield Jackson Atlanta International Airport,United States,216,217
5,ORD,Chicago O'Hare International Airport,United States,203,206
6,PEK,Beijing Capital International Airport,China,206,206
7,MUC,Munich Airport,Germany,189,191
8,DFW,Dallas Fort Worth International Airport,United States,185,187
9,DME,Domodedovo International Airport,Russia,188,187


Вплив `resetProbability`: топ-5 аеропортів для різних значень.

In [27]:
top_by_reset = {}
for reset in (0.05, 0.15, 0.3, 0.5, 0.85):
    top = g.pageRank(resetProbability=reset, maxIter=20).vertices.orderBy(F.desc("pagerank")).limit(5)
    top_by_reset[reset] = [f"{row.code} ({row.pagerank:.1f})" for row in top.collect()]

pd.DataFrame(top_by_reset, index=range(1, 6)).T.rename_axis("resetProbability")

,1,2,3,4,5
resetProbability,,,,,
0.05,ATL (17.7),ISL (16.7),CDG (16.6),FRA (16.6),ORD (16.5)
0.15,ATL (16.0),ORD (14.6),ISL (14.6),DEN (14.5),DFW (14.3)
0.30,ATL (13.8),DEN (13.7),DME (13.1),DFW (12.8),ORD (12.5)
0.50,DEN (11.6),DME (11.5),ATL (11.0),DFW (10.5),ORD (9.9)
0.85,DME (5.2),DEN (5.1),ATL (4.6),DFW (4.6),ORD (4.2)


## Завдання 5. Кластеризація та аналіз зв'язності

### Компоненти зв'язності

Слабко зв'язні компоненти (напрямок рейсів ігнорується). Алгоритм потребує checkpoint-директорії, спільної для всіх workers.

In [28]:
components = g.connectedComponents().cache()
component_sizes = components.groupBy("component").count().orderBy(F.desc("count"))
show(component_sizes)

,component,count
0,1,3303
1,1998,10
2,3726,4
3,5642,4
4,3860,4
5,7309,2
6,6448,2


Кластери з не менше ніж 5 аеропортів:

In [29]:
show(component_sizes.filter("count >= 5"))

,component,count
0,1,3303
1,1998,10


Ізольовані підграфи: групи з 2 і більше аеропортів, не пов'язані з основною мережею (найбільшою компонентою):

In [30]:
main_component = component_sizes.first().component
isolated = (
    components.join(component_sizes, "component")
    .filter((F.col("component") != main_component) & (F.col("count") >= 2))
    .groupBy("component", "count")
    .agg(
        F.concat_ws(", ", F.sort_array(F.collect_list(F.concat_ws(" ", "code", F.concat(F.lit("("), "city", F.lit(")")))))).alias("airports"),
        F.concat_ws(", ", F.array_distinct(F.collect_list("country"))).alias("countries"),
    )
    .orderBy(F.desc("count"))
)
show(isolated)

,component,count,airports,countries
0,1998,10,"BMY (Waala), GEA (Noumea), ILP (Île des Pins), KNQ (Kone), KOC (Koumac), LIF...",New Caledonia
1,3726,4,"BFI (Seattle), CLM (Port Angeles), ESD (Eastsound), FRD (Friday Harbor)",United States
2,5642,4,"ERS (Windhoek), MPA (Mpacha), NDU (Rundu), OND (Ondangwa)",Namibia
3,3860,4,"AKB (Atka), DUT (Unalaska), IKO (Nikolski), KQA (Akutan)",United States
4,7309,2,"SPB (Charlotte Amalie), SSB (Christiansted)",Virgin Islands
5,6448,2,"BLD (Boulder City), GCW (Grand Canyon West)",United States


### Label Propagation

Компоненти зв'язності розділяють лише непов'язані частини мережі, тож майже всі аеропорти потрапляють в одну компоненту. Label Propagation виділяє спільноти всередині зв'язної мережі: аеропорти, тісніше пов'язані між собою, ніж з рештою.

In [31]:
communities = g.labelPropagation(maxIter=5).cache()
community_sizes = communities.groupBy("label").count()
community_sizes.filter("count >= 5").count()

71

Спільноти з не менше ніж 5 аеропортів: розмір, кількість країн, головна країна та найбільші аеропорти спільноти:

In [32]:
by_degree = Window.partitionBy("label").orderBy(F.desc("outDegree"))
by_count = Window.partitionBy("label").orderBy(F.desc("count"))

hubs = (
    communities.join(degrees, "id")
    .withColumn("rank", F.row_number().over(by_degree))
    .filter("rank <= 5")
    .groupBy("label")
    .agg(F.concat_ws(", ", F.collect_list("code")).alias("main_airports"))
)
main_country = (
    communities.groupBy("label", "country").count()
    .withColumn("rank", F.row_number().over(by_count))
    .filter("rank = 1")
    .select("label", F.col("country").alias("main_country"), F.col("count").alias("main_country_airports"))
)
community_summary = (
    communities.groupBy("label")
    .agg(F.count("*").alias("airports"), F.countDistinct("country").alias("countries"))
    .filter("airports >= 5")
    .join(main_country, "label")
    .join(hubs, "label", "left")
    .withColumn("main_country_share", F.round(F.col("main_country_airports") / F.col("airports"), 2))
    .drop("main_country_airports")
    .orderBy(F.desc("airports"))
)
show(community_summary, 15)

,label,airports,countries,main_country,main_airports,main_country_share
0,193,796,85,United States,"ISL, ATL, DXB, JFK, EWR",0.43
1,16,351,50,Norway,"FRA, CDG, AMS, MUC, LHR",0.12
2,302,336,64,France,"BCN, MAD, PMI, AGP, MXP",0.10
3,340,216,33,India,"DME, BKK, VKO, SVX, TAS",0.25
4,156,159,20,Japan,"PEK, PVG, CAN, HKG, ICN",0.35
5,5,157,17,Australia,"SIN, KUL, SYD, MNL, CGK",0.46
6,49,102,3,United States,"ORD, DFW, DEN, IAH, LAX",0.49
7,3364,85,3,China,"WUX, TXN, CZX, XUZ, MIG",0.98
8,2531,73,1,Brazil,"BSB, CNF, SSA, CGH, CWB",1.00
9,2276,70,9,China,"URC, CGO, KWE, FOC, NNG",0.63


## Висновки

- **Дані.** З 67 663 маршрутів координати отримали 67 235: відкинуто маршрути без ідентифікатора аеропорту, петлю та маршрути через пошкоджений запис аеропорту 5613 (координати Каліфорнії для аеропорту в Замбії).
- **Кілометраж.** Найбільша сумарна довжина власних (не codeshare) маршрутів — у Ryanair: 3,70 млн км на 2 484 маршрутах; найменша — у TransHolding System (Бразилія): 36 км на 2 маршрутах.
- **Маршрути Німеччина → Велика Британія.** Мотиви знаходять 73 прямі маршрути, 9 178 з однією пересадкою і 506 498 з двома. BFS повертає лише найкоротші за кількістю сегментів шляхи — 73 прямі. Найчастіші пересадки — курортні аеропорти Іспанії (PMI, AGP, TFS), куди літають лоукостери з обох країн.
- **TXL → EDI.** Прямого рейсу немає; найкоротший маршрут TXL → AMS → EDI (1 245 км), найдовший з не більше ніж 3 пересадками — TXL → ORD → HKG → EWR → EDI (37 808 км).
- **PageRank.** Найвпливовіші — ATL, ORD, ISL, DEN, DFW: хаби з великою кількістю напрямків, на які ведуть рейси з інших впливових аеропортів. За кількістю напрямків лідирують FRA, CDG, AMS. Зі зростанням `resetProbability` значення вирівнюються, а до топу піднімаються аеропорти з багатьма локальними напрямками (DME, DEN).
- **Зв'язність.** 3 303 аеропорти утворюють одну компоненту; 6 ізольованих груп (Нова Каледонія, Намібія, Алеутські острови та ін.) мають лише внутрішні рейси. Label Propagation виділяє 71 спільноту з не менше ніж 5 аеропортів, що відповідають регіональним мережам.
